## Install Libraries

Runs the **retrieval stage of the baseline**: encode each gold mention
**together with its sentence context**, scan the KB vectors built by
`build_KB.ipynb`, aggregate surface hits into concepts and save the top-n
candidates. No LLM feedback, no fusion, no re-ranking — selecting among the n
candidates is stage 2, in `baseline/e2e.ipynb`.

Output is `BASELINE_CANDIDATES_PATH`, a JSONL handover file. Only
`source/common/` is used; nothing from `source/GRF/`.

`tqdm` is installed explicitly for the progress bars.

In [1]:
%pip install "tqdm>=4.66" "sentence-transformers>=2.7"

## Import Libraries

Third-party imports only. The shared `common` package lives inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.

In [2]:
from __future__ import annotations

import gc
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()

Python: 3.13.16
tqdm: 4.67.3


## Constants and Configuration

All paths and retrieval parameters live here.

Drive layout:

- `COMMON_DIR/` — shared root (`DRIVE_ROOT`),
- `COMMON_DIR/data/` — shared inputs: the KB artifact from `build_KB.ipynb` and
  the gold JSONL,
- `COMMON_DIR/output/{METHOD}/` — this method's outputs (`METHOD = "baseline"`).

The baseline reads only the KB artifact and the gold data; the KB CSV is not
needed because no concept names or LLM feedback are used at this stage.

`ENCODER_MODEL` is a HuggingFace id or a local model directory; it must match
the checkpoint recorded in the artifact.

Retrieval keeps the mention in place in its sentence, marks it with `[SEP]` and
uses its pooled token states **plus its own surface embedding** (`common.query`
+ `SentenceTransformerEncoder.encode_mention_in_context`), the same query
`GRF.pipeline` builds for `Q_0`. `TOP_K_CANDIDATE` is the n saved for
`baseline/e2e.ipynb`.

In [3]:
# ---------------------------------------------------------------------------
# Git repository
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive layout
#   COMMON_DIR/                     shared root
#   COMMON_DIR/data/                shared inputs (KB artifact, gold data)
#   COMMON_DIR/output/{METHOD}/     this method's outputs
# ---------------------------------------------------------------------------
METHOD = "baseline"
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
COMMON_DIR = DRIVE_ROOT / "work" / "nlp" / "UMLS" / "entity_linking"
DRIVE_DATA_DIR = COMMON_DIR / "data"
OUTPUT_DIR = COMMON_DIR / "output" / METHOD

KB_ARTIFACT_PATH = DRIVE_DATA_DIR / "grf_kb.npz"                # from build_KB.ipynb
GOLD_DATA_PATH = DRIVE_DATA_DIR / "translated.jsonl"                  # gold BEL JSONL
BASELINE_CANDIDATES_PATH = OUTPUT_DIR / "baseline_candidates.jsonl"

# ---------------------------------------------------------------------------
# Encoder (HuggingFace id or a local directory; must match the artifact)
# ---------------------------------------------------------------------------
ENCODER_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
ENCODER_BATCH_SIZE = 32
DEVICE = "cuda"                # "cuda" (Colab GPU) or "cpu"

# ---------------------------------------------------------------------------
# Dense retrieval (no feedback)
# ---------------------------------------------------------------------------
USE_CONTEXT_QUERY = True        # True = mention pooled in its sentence; False = encode(mention) alone
TOP_K_RETRIEVAL = 30            # surface-form hits; enough to aggregate n concepts
TOP_K_CANDIDATE = 10            # n candidates saved for e2e.ipynb
ENCODE_BATCH_SIZE = 64          # mentions encoded per progress step
INDEX_KB = None                 # optional source_kb filter; None = all entries

# ---------------------------------------------------------------------------
# Evaluation / runtime
# ---------------------------------------------------------------------------
MAX_SAMPLES = None              # None = all mentions; an int truncates for smoke runs
CACHE_DIR = Path("/content/baseline_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value


log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"KB artifact  : {KB_ARTIFACT_PATH}")
print(f"Gold data    : {GOLD_DATA_PATH}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Candidates   : {BASELINE_CANDIDATES_PATH}")
print(f"Query        : {'mention + context' if USE_CONTEXT_QUERY else 'mention only'}")
print(f"Encoder      : {ENCODER_MODEL}")

[18:53:11 +   0.0s] Configuration loaded.
Repository   : https://github.com/minhhhduc/bio-linking-.git (main)
KB artifact  : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/grf_kb.npz
Gold data    : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/translated.jsonl
Output dir   : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/baseline
Candidates   : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/baseline/baseline_candidates.jsonl
Query        : mention + context
Encoder      : cambridgeltl/SapBERT-from-PubMedBERT-fulltext


## Stage 1 — Clone or Update the Code Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/`. Re-running this cell
updates the existing clone instead of downloading it again.

In [4]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from common.aggregation import aggregate_hits
from common.candidates import load_candidates, save_candidates
from common.data import load_bel_jsonl
from common.encoder import SentenceTransformerEncoder
from common.evaluator import accuracy_at_k, metrics_from_responses
from common.index import SINGLE_FILE_FORMAT, VectorIndex
from common.schemas import Response

log(f"Repository ready at {REPO_CLONE_DIR}; common imported from {SOURCE_DIR}.")
log(f"Single-file artifact format: {SINGLE_FILE_FORMAT}")

[18:53:11 +   0.0s] Cloning https://github.com/minhhhduc/bio-linking-.git (branch 'main') into /content/bio_linking ...
[18:53:11 +   0.0s] git clone --depth 1 --branch main https://github.com/minhhhduc/bio-linking-.git /content/bio_linking
[18:53:13 +   1.9s] Repository ready at /content/bio_linking; common imported from /content/bio_linking/source.
[18:53:13 +   1.9s] Single-file artifact format: grf-vector-index-npz-v1


## Stage 2 — Mount Google Drive and Locate Inputs

The KB artifact and the gold data live on Drive, not in the repository. This
stage mounts Drive and checks both inputs plus the single candidate output path
before any model is loaded.

In [5]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

artifact_path = Path(require_configured(KB_ARTIFACT_PATH, "KB_ARTIFACT_PATH"))
gold_path = Path(require_configured(GOLD_DATA_PATH, "GOLD_DATA_PATH"))
candidates_path = Path(
    require_configured(BASELINE_CANDIDATES_PATH, "BASELINE_CANDIDATES_PATH")
)

for label, path in (
    ("KB artifact", artifact_path),
    ("Gold data", gold_path),
):
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    log(f"{label}: {path} ({path.stat().st_size / 1024 ** 2:.2f} MB)")

if candidates_path.suffix.lower() != ".jsonl":
    raise ValueError(
        f"BASELINE_CANDIDATES_PATH must end in .jsonl, got {candidates_path.name!r}."
    )
log(f"Candidate output: {candidates_path}")

Mounted at /content/drive
[18:54:25 +  73.9s] KB artifact: /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/grf_kb.npz (9231.94 MB)
[18:54:25 +  73.9s] Gold data: /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/translated.jsonl (0.22 MB)
[18:54:25 +  73.9s] Candidate output: /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/baseline/baseline_candidates.jsonl


## Stage 3 — Load the KB Artifact and the Encoder

`VectorIndex.load_npz` restores the vectors and metadata from the single file.
The encoder model itself is not stored in the artifact, so this stage loads the
same checkpoint recorded in `encoder_info` and checks the dimensions agree.
With `DEVICE = "cuda"`, `index.use_gpu()` keeps one GPU copy of the vectors:
retrieval is then a batched matrix multiply on the GPU, and only the top-k rows
cross back to CPU.

In [6]:
log(f"Loading KB artifact {artifact_path} ...")
index = VectorIndex.load_npz(artifact_path)
log(f"Artifact loaded: {len(index):,} entries x {index.vectors.shape[1]} dims.")

stored_encoder = (index.encoder_info or {}).get("model")
if stored_encoder is None:
    log("WARNING: artifact has no encoder_info; assuming ENCODER_MODEL is the same checkpoint.")
elif stored_encoder != ENCODER_MODEL:
    raise ValueError(
        f"Artifact was built with encoder {stored_encoder!r}, but ENCODER_MODEL is "
        f"{ENCODER_MODEL!r}. Set ENCODER_MODEL to the encoder used by build_KB.ipynb."
    )

if DEVICE == "cuda":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "DEVICE='cuda' but no CUDA device is available. In Colab: "
            "Runtime > Change runtime type > Hardware accelerator > GPU."
        )
    log(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    log("Running on CPU (DEVICE='cpu').")

log(f"Loading encoder {ENCODER_MODEL!r} on {DEVICE} ...")
encoder = SentenceTransformerEncoder(
    ENCODER_MODEL, batch_size=ENCODER_BATCH_SIZE, device=DEVICE
)
if encoder.dim != index.vectors.shape[1]:
    raise ValueError(
        f"Encoder dimension {encoder.dim} does not match artifact dimension "
        f"{index.vectors.shape[1]}."
    )
log(f"Encoder ready: dim={encoder.dim}.")

if DEVICE == "cuda":
    try:
        index.use_gpu(DEVICE)
    except RuntimeError as exc:
        torch.cuda.empty_cache()
        log(f"WARNING: GPU search unavailable ({exc}); using the CPU scan.")
    else:
        log(f"Search device: {index.search_device} (exact cosine on GPU).")
if index.search_device == "cpu":
    log("Search device: cpu (exact cosine on CPU).")

[18:54:25 +  73.9s] Loading KB artifact /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/grf_kb.npz ...
[18:58:30 + 318.4s] Artifact loaded: 3,342,067 entries x 768 dims.
[18:58:33 + 322.3s] GPU: Tesla T4
[18:58:33 + 322.3s] Loading encoder 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext' on cuda ...


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[18:59:02 + 350.6s] Encoder ready: dim=768.


/content/bio_linking/source/common/encoder.py:60: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  declared = self._model.get_sentence_embedding_dimension()


[18:59:04 + 352.7s] Search device: cuda (exact cosine on GPU).


## Stage 4 — Load and Validate the Gold Data

`load_bel_jsonl` validates structure and token spans. Structural problems are
reported as `DataIssue` entries; span issues never drop a sample. All samples
are used for candidate retrieval. The loader reports each record through
`progress_callback`, which drives the bar.

In [7]:
log(f"Loading gold BEL data from {gold_path} ...")
started = time.time()
with progress_bar(None, "Loading gold records", "record") as report:
    loaded = load_bel_jsonl(gold_path, strict=False, progress_callback=report)
samples = list(loaded.samples)
if MAX_SAMPLES is not None:
    if isinstance(MAX_SAMPLES, bool) or not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES < 1:
        raise ValueError("MAX_SAMPLES must be None or a positive integer.")
    samples = samples[:MAX_SAMPLES]

log(
    f"Loaded {len(loaded.samples):,} mention samples "
    f"({len(samples):,} selected) in {time.time() - started:.1f}s."
)
if loaded.issues:
    log(f"WARNING: {len(loaded.issues):,} data issue(s); first: {loaded.issues[0]}")
if not samples:
    raise ValueError(f"{gold_path} produced no usable mention samples.")

[18:59:04 + 352.8s] Loading gold BEL data from /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/translated.jsonl ...


Loading gold records: 0record [00:00, ?record/s]

[18:59:05 + 354.0s] Loaded 959 mention samples (959 selected) in 1.3s.


## Stage 5 — Dense Retriever (mention pooled in place inside its context)

Each sample's sentence is tokenized as written, with `[SEP]` markers inserted
around the mention at its gold `start_token`/`end_token` position
(`common.query.mention_query`). The query vector is the **mean of the mention's
own token states between the markers** added to **`encoder.encode(mention)`** —
`encoder.encode_mention_in_context`. The words before and after the mention
stay context: they reach it through attention but are not averaged into its
vector. The `encode(mention)` term is the same function that built every index
row, so the query keeps the exact-surface signal; the contextual mean alone
lives in a different region of the space than the surface it must hit.

`USE_CONTEXT_QUERY = False` skips all of that and calls
`encoder.encode(mention)` — the library's own pooling, no context, no manual
token selection.

`GRF.pipeline` builds `Q_0` with the same function and the same switch, so both
methods retrieve with an identical query. There is no feedback call, no fusion
weight and no LLM client here.

Two phases: **encode all** mentions (progress bar), then **one batched scan**
`R = Q @ A.T` over the whole query set (`index.search_batch`). On GPU the
matmul and `topk` run on the device in query chunks and only the top-k rows
come back to CPU, so scoring does not loop per mention in Python.

`aggregate_hits` keeps the full untruncated concept ranking for R@5 and
truncates only the n candidates saved for `e2e.ipynb`.

In [8]:
if TOP_K_CANDIDATE < 5:
    log("WARNING: TOP_K_CANDIDATE < 5, so e2e acc@5 cannot exceed n.")
if TOP_K_RETRIEVAL < TOP_K_CANDIDATE:
    log("WARNING: TOP_K_RETRIEVAL < TOP_K_CANDIDATE, so fewer than n concepts may be saved.")


def dense_responses(samples, report):
    """Encode every mention, then score the whole query set in one scan."""
    chunks = []
    for start in range(0, len(samples), ENCODE_BATCH_SIZE):
        batch = samples[start : start + ENCODE_BATCH_SIZE]
        mentions = [sample.mention for sample in batch]
        if USE_CONTEXT_QUERY:
            vectors = encoder.encode_mention_in_context(
                mentions,
                [sample.context for sample in batch],
                [(sample.start_token, sample.end_token) for sample in batch],
            )
        else:
            vectors = encoder.encode(mentions)
        chunks.append(vectors)
        report(start + len(batch), len(samples))

    queries = np.concatenate(chunks, axis=0)
    log(
        f"Encoded {queries.shape[0]:,} queries x {queries.shape[1]} dims; "
        f"scoring on {index.search_device} ..."
    )
    hit_lists = index.search_batch(queries, TOP_K_RETRIEVAL, source_kb=INDEX_KB)

    responses = []
    for sample, hits in zip(samples, hit_lists):
        ranking = aggregate_hits(hits)
        if not ranking:
            raise ValueError(
                f"No candidate for {sample.mention!r} "
                f"(record {sample.record_index}, entity {sample.entity_index})."
            )
        candidates = tuple(ranking[:TOP_K_CANDIDATE])
        responses.append(
            Response(
                cui=candidates[0].cui,
                candidates=candidates,
                debug={
                    "concept_ranking": [
                        {
                            "cui": candidate.cui,
                            "score": candidate.score,
                            "surface": candidate.surface,
                        }
                        for candidate in ranking
                    ],
                    "score_kind": "embedding",
                },
            )
        )
    return responses


log(
    f"Dense retriever ready: {len(index):,} entries; "
    f"top_k_retrieval={TOP_K_RETRIEVAL} n_candidates={TOP_K_CANDIDATE} "
    f"encode_batch={ENCODE_BATCH_SIZE} mode="
    f"{'mention + context' if USE_CONTEXT_QUERY else 'mention only'}."
)

[18:59:05 + 354.0s] Dense retriever ready: 3,342,067 entries; top_k_retrieval=30 n_candidates=10 encode_batch=64 mode=mention + context.


## Stage 6 — Run Dense Retrieval

One encode-and-search pass produces, per mention, the Top-1 prediction, the
candidate list and the untruncated concept ranking. The batch loop reports each
finished chunk through the progress bar.

In [9]:
with progress_bar(len(samples), "Dense retrieval", "mention") as report:
    started = time.time()
    responses = dense_responses(samples, report)

log(
    f"Retrieved candidates for {len(responses):,} mentions "
    f"in {time.time() - started:.1f}s."
)

Dense retrieval:   0%|          | 0/959 [00:00<?, ?mention/s]

[18:59:09 + 358.2s] Encoded 959 queries x 768 dims; scoring on cuda ...
[18:59:18 + 367.0s] Retrieved candidates for 959 mentions in 13.0s.


## Stage 7 — Save Candidates to One File

`save_candidates` writes one JSONL record per mention to
`BASELINE_CANDIDATES_PATH`: the sample fields, the n candidates and the
untruncated concept ranking. `baseline/e2e.ipynb` reads this file. It is
reloaded immediately to prove it is usable.

In [10]:
log(f"Saving candidates to {candidates_path} ...")
started = time.time()
with progress_bar(len(samples), "Saving candidate records", "record") as report:
    save_candidates(candidates_path, samples, responses, progress_callback=report)
log(
    f"Saved {len(responses):,} records "
    f"({candidates_path.stat().st_size / 1024 ** 2:.2f} MB) "
    f"in {time.time() - started:.1f}s."
)

log("Reloading the candidate file to verify it ...")
with progress_bar(None, "Reloading candidate records", "record") as report:
    reloaded_samples, reloaded_responses = load_candidates(
        candidates_path, progress_callback=report
    )
assert len(reloaded_samples) == len(samples), (len(reloaded_samples), len(samples))
assert len(reloaded_responses) == len(responses), (len(reloaded_responses), len(responses))
assert [sample.gold_cui for sample in reloaded_samples] == [
    sample.gold_cui for sample in samples
]
assert [response.candidates for response in reloaded_responses] == [
    response.candidates for response in responses
]
assert [
    response.debug.get("concept_ranking") or []
    for response in reloaded_responses
] == [
    response.debug.get("concept_ranking") or []
    for response in responses
]
log(f"Verified: {len(reloaded_samples):,} samples and responses reloaded.")

[18:59:18 + 367.0s] Saving candidates to /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/baseline/baseline_candidates.jsonl ...


Saving candidate records:   0%|          | 0/959 [00:00<?, ?record/s]

[18:59:19 + 367.8s] Saved 959 records (3.80 MB) in 0.8s.
[18:59:19 + 367.8s] Reloading the candidate file to verify it ...


Reloading candidate records: 0record [00:00, ?record/s]

[18:59:19 + 368.0s] Verified: 959 samples and responses reloaded.


## Stage 8 — Retrieval and Candidate Accuracy

Two metric groups over the same samples:

- **R@1/R@5** from the untruncated concept ranking (`metrics_from_responses`),
- **acc@1/acc@5** = share of mentions whose gold CUI is among the first k
  **saved candidates** (`accuracy_at_k`): `acc@k = (1/n) Σ 1[gold_i ∈ top-k candidates_i]`.

No re-ranking happens here, so the saved candidate list is the retrieval
ranking from its first entry: acc@k equals R@k for k ≤ `TOP_K_CANDIDATE`.
`baseline/e2e.ipynb` reports acc@k again on the LLM-reranked candidate list.

In [11]:
overall, by_label = metrics_from_responses(samples, responses)
acc1_overall, acc1_by_label = accuracy_at_k(samples, responses, 1)
acc5_overall, acc5_by_label = accuracy_at_k(samples, responses, 5)


def _row(label, metrics, acc1, acc5) -> str:
    return (
        f"{str(label):<16}{metrics.n:>7}"
        f"{100 * metrics.recall_at_1:>10.2f}{100 * metrics.recall_at_5:>10.2f}"
        f"{100 * acc1:>10.2f}{100 * acc5:>10.2f}"
    )


print(
    f"{'label':<16}{'n':>7}{'R@1 %':>10}{'R@5 %':>10}"
    f"{'acc@1 %':>10}{'acc@5 %':>10}"
)
print(_row("ALL", overall, acc1_overall, acc5_overall))
for label, metrics in by_label.items():
    print(_row(label, metrics, acc1_by_label[label], acc5_by_label[label]))

log("Dense retrieval complete. Candidate file is ready for e2e.ipynb.")

label                 n     R@1 %     R@5 %   acc@1 %   acc@5 %
ALL                 959     55.79     73.31     55.79     73.31
DISEASE             698     58.17     75.64     58.17     75.64
SYMPTOM             189     52.38     68.25     52.38     68.25
DIAGNOSTIC           21     28.57     76.19     28.57     76.19
CAUSE                26     34.62     42.31     34.62     42.31
TREATMENT            25     60.00     76.00     60.00     76.00
[18:59:19 + 368.1s] Dense retrieval complete. Candidate file is ready for e2e.ipynb.


## Stage 9 — Release Memory

Candidates are saved and the retrieval metrics are printed, so drop the vector
index (including its GPU copy), the encoder and the loaded samples/responses,
then empty the CUDA cache. `baseline/e2e.ipynb` reads only the saved file.


In [12]:
if "index" in globals():
    index.release_gpu()

for name in (
    "index",
    "encoder",
    "loaded",
    "samples",
    "responses",
    "reloaded_samples",
    "reloaded_responses",
):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, total_b = torch.cuda.mem_get_info()
        log(
            f"CUDA cache emptied: {free_b / 1024 ** 3:.2f}/"
            f"{total_b / 1024 ** 3:.2f} GiB free."
        )
except ImportError:
    pass

log("Baseline retrieval memory released (index, encoder, samples, responses).")

[18:59:20 + 368.6s] CUDA cache emptied: 14.41/14.56 GiB free.
[18:59:20 + 368.6s] Baseline retrieval memory released (index, encoder, samples, responses).
